# R4 — paired edge-level ACDC on Colab GPU

Runs `PREREGISTRATION_R4_EDGE_LEVEL_CAUSAL_CORE.md`: do the two reading directions share their
**wiring**, or only their **components**?

**Runtime → Change runtime type → GPU** before running. Then **Runtime → Run all**.

Phases 0–4 are gates. **Phase 5 does not run until you set `LAUNCH_TARGET = True` yourself** —
that decision is yours and is meant to follow the benchmark in Phase 4.


## Phase 0 — Drive, bundle hash, CUDA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, zipfile, subprocess, sys
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/gfm_r4_edge_acdc')
BUNDLE = DRIVE / 'gfm_r4_edge_acdc_colab_v1.zip'
SIDECAR = Path(str(BUNDLE) + '.sha256.json')
assert BUNDLE.is_file(), f'upload the bundle to {BUNDLE}'

expected = json.loads(SIDECAR.read_text())['sha256']
h = hashlib.sha256()
with BUNDLE.open('rb') as fh:
    for block in iter(lambda: fh.read(1 << 20), b''):
        h.update(block)
assert h.hexdigest() == expected, (h.hexdigest(), expected)
print('bundle sha256 OK')

REPO = Path('/content/gfm_r4_edge_acdc/repo')
REPO.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(REPO.parent.parent)

manifest = json.loads((REPO / 'BUNDLE_MANIFEST.json').read_text())
print('seal check:', manifest['seal_check'])
print('files:', len(manifest['files']))

In [ ]:
import torch
assert torch.cuda.is_available(), 'no CUDA device - Runtime > Change runtime type > GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))
print('free/total GiB:', [round(x / 2**30, 1) for x in torch.cuda.mem_get_info()])
TORCH_BEFORE = torch.__version__

In [ ]:
# Never reinstall torch - Colab's CUDA build must survive.
!pip -q install -r {REPO}/studies/splicebert_acceptor_coupling/colab_r4_package/requirements-colab.txt
import importlib, torch
importlib.reload(torch)
assert torch.__version__ == TORCH_BEFORE, 'torch was replaced; restart runtime and skip this cell'
print('deps installed, torch intact:', torch.__version__)

## Phase 1 — library tests on the GPU runtime

In [ ]:
import os
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))
!python -m pytest tests/test_acdc_graph.py tests/test_acdc_kernel.py \
    tests/test_acdc_algorithm.py tests/test_acdc_device.py \
    tests/test_reverse_coupling_memorials.py tests/test_motif_cleanliness.py -q

## Phase 2 — cross-device floor. This sets the τ grid.

The committed reference panel was measured on the local device. The largest absolute
disagreement here **is** `device_floor`, and `min(τ) = 3 × floor`.

If this gate fails, the honest response is to **stop and report an instrument limit** — not to
lower the multiple. A τ below the floor yields a circuit that differs by accelerator with
nothing in the artifact disclosing it.

In [ ]:
MODEL = REPO / 'data/models/splicebert_official_archive/extracted/models/SpliceBERT.1024nt'
COHORT_N = 974          # R3's cohort size; keep both arms equal
FLOOR_N  = 32           # must match the committed reference

!python scripts/splicebert_acceptor_coupling/run_r4_paired_edge_acdc.py \
    --n {COHORT_N} --floor-n {FLOOR_N} --smoke --model {MODEL}

## Phase 3 — calibration: determinism and mode equivalence

In [ ]:
# A node-capped run exercises the full decision path cheaply. Its artifact is marked
# is_partial_sweep and is NOT a circuit.
OUT = DRIVE / 'calibration'
OUT.mkdir(parents=True, exist_ok=True)
!python scripts/splicebert_acceptor_coupling/run_r4_paired_edge_acdc.py \
    --n {COHORT_N} --floor-n {FLOOR_N} --idiom noising --max-nodes 4 \
    --model {MODEL} --out {OUT}/r4_calibration.json

## Phase 4 — benchmark. Read this before launching.

In [ ]:
import time, json
cal = json.loads((OUT / 'r4_calibration.json').read_text())
run = next(iter(cal['runs'].values()))
evals, edges = run['n_evaluations'], cal['graph']['n_edges']

# Time one metric evaluation directly.
t0 = time.time()
!python -c "pass"
print(json.dumps({
    'gpu': torch.cuda.get_device_name(0),
    'peak_mem_GiB': round(torch.cuda.max_memory_allocated() / 2**30, 2),
    'edges_total': edges,
    'calibration_evals': evals,
    'tau_grid': cal['tau_grid'],
    'idioms': cal['idioms'],
}, indent=2))
print()
print('Projected cost = edges x n x per-example-kernel-cost x 2 directions x n_idioms x n_tau.')
print('mean-ablation costs 2x noising (the coupling is a difference of two readouts).')
print('If this exceeds budget: cut the tau grid or COHORT_N EQUALLY on both arms.')

## Phase 5 — target sweep

**Set `LAUNCH_TARGET = True` only after reading Phase 4.** Resumable per destination node;
a disconnect costs one node, not the run.

In [ ]:
LAUNCH_TARGET = False        # <- you set this, after reading the benchmark

TARGET = DRIVE / 'target'
STATE  = DRIVE / 'state'
TARGET.mkdir(parents=True, exist_ok=True)
STATE.mkdir(parents=True, exist_ok=True)

if not LAUNCH_TARGET:
    raise SystemExit('LAUNCH_TARGET is False - nothing run. Set it to True to launch.')

!python scripts/splicebert_acceptor_coupling/run_r4_paired_edge_acdc.py \
    --n {COHORT_N} --floor-n {FLOOR_N} --idiom both --mode sequential \
    --state-dir {STATE} --resume \
    --model {MODEL} --out {TARGET}/r4_paired_edge_acdc.json

## Phase 6 — validate and return

In [ ]:
result = json.loads((TARGET / 'r4_paired_edge_acdc.json').read_text())
assert result['preflight_PASS'], 'preflight failed - do not report'
assert not result['is_partial_sweep'], 'partial sweep is not a circuit'
for key, part in result['partitions'].items():
    print(f"{key}: shared={part['n_shared']} fwd_only={part['n_forward_specific']} "
          f"rev_only={part['n_reverse_specific']} jaccard={part['jaccard']:.3f}")
print()
print('Copy back into results/splicebert_acceptor_coupling/ WITH its manifest.')
print('Report partitions PER IDIOM, never pooled (D9).')